In [1]:
import numpy as np
from sklearn.datasets import fetch_openml
from tqdm import tqdm
mnist = fetch_openml('mnist_784', version=1, as_frame=False, parser='liac-arff')

In [4]:
# --- Hyperparameters ---
NUM_INPUT = 784
NUM_OUTPUT = 500
DT = 0.005         # 5 ms time step
T_MAX = 0.350      # 350 ms per image
LR = 0.001         # Learning rate
EPOCHS = 5
NUM_IMAGES = 60000

def train_snn():
    print("Loading MNIST training data via sklearn...")
    
    # Use the first 60,000 images for training and normalize
    images = mnist.data[:NUM_IMAGES] / 255.0
    
    # Initialize weights U(0, 1)
    weights = np.random.uniform(0, 1, (NUM_INPUT, NUM_OUTPUT))
    
    # Pre-allocate neuron states
    v_pre = np.zeros(NUM_INPUT)
    v_post = np.zeros(NUM_OUTPUT)
    n_adapt = np.zeros(NUM_OUTPUT)
    
    refractory_pre = np.zeros(NUM_INPUT)
    refractory_post = np.zeros(NUM_OUTPUT)
    wta_clamp = np.zeros(NUM_OUTPUT)

    print("Starting training...")
    for epoch in range(EPOCHS):
        print(f"Epoch {epoch+1}:")
        for img_idx, img in tqdm(enumerate(images), total=len(images), desc="training: "):
            # Reset states for new image
            # v_pre.fill(0.0)
            # v_post.fill(0.0)
            
            # Input current proportional to pixel intensity
            I_in = img 
            
            for t in np.arange(0, T_MAX, DT):
                
                refractory_pre -= DT
                refractory_post -= DT
                wta_clamp -= DT
                
                # --- Update Input Neurons (LIF) ---
                active_pre = refractory_pre <= 0
                v_pre[active_pre] += (DT / 0.03) * (-v_pre[active_pre] + I_in[active_pre] + 0.5)
                
                spikes_pre = v_pre >= 1.0
                v_pre[spikes_pre] = -1.0
                refractory_pre[spikes_pre] = 0.005
                
                # --- Update Output Neurons (ALIF) ---
                active_post = (refractory_post <= 0) & (wta_clamp <= 0)
                I_post = spikes_pre.astype(float) @ weights
                
                v_post[active_post] += (DT / 0.03) * (-v_post[active_post] + I_post[active_post] - n_adapt[active_post])
                n_adapt -= (DT / 1.0) * n_adapt
                
                spikes_post = v_post >= 1.0
                
                # --- Apply WTA and VDSP ---
                if spikes_post.any():
                    winner_idx = np.argmax(v_post) # Winner takes all
                    v_post[winner_idx] = 0.0
                    refractory_post[winner_idx] = 0.005
                    n_adapt[winner_idx] += 0.01
                    
                    # Clamp others
                    wta_clamp[:] = 0.010
                    wta_clamp[winner_idx] = 0.0
                    v_post[wta_clamp > 0] = 0.0
                    
                    # VDSP Weight Update for winner
                    v_pre_winner = v_pre
                    
                    # Potentiation (V_pre < 0)
                    pot_mask = v_pre_winner < 0
                    weights[pot_mask, winner_idx] += LR * (1.0 - weights[pot_mask, winner_idx]) * (-v_pre_winner[pot_mask])
                    
                    # Depression (V_pre > 0)
                    dep_mask = v_pre_winner >= 0
                    weights[dep_mask, winner_idx] -= LR * weights[dep_mask, winner_idx] * (1.0 - v_pre_winner[dep_mask])
                    
                    # Clip weights (safety bound)
                    weights[:, winner_idx] = np.clip(weights[:, winner_idx], 0, 1)
                                
    np.save('vdsp_weights.npy', weights)
    print("Training complete. Weights saved.")

In [5]:
train_snn()

Loading MNIST training data via sklearn...
Starting training...
Epoch 1:


training: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████| 60000/60000 [20:20<00:00, 49.15it/s]


Epoch 2:


training: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████| 60000/60000 [18:12<00:00, 54.91it/s]


Epoch 3:


...

Epoch 4:


training: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████| 60000/60000 [18:26<00:00, 54.20it/s]


Epoch 5:


training: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████| 60000/60000 [18:16<00:00, 54.72it/s]

Training complete. Weights saved.
